# *FTheoryTools: Advancing Computational Capabilities for F-Theory Research*

* Authors: Martin Bies, Miķelis Emīls Miķelsons, Andrew P. Turner
* Version: OSCAR version 1.5 or newer.

This tutorial collects the Julia code used in the publication "FTheoryTools: Advancing Computational Capabilities for F-Theory Research" (https://arxiv.org/abs/2506.13849). As such, this tutorial provides an extensive survey of almost all of the functionality of FTheoryTools.

### Table of Contents

- [3. Algorithmic Framework for F-Theory Models in OSCAR](#3:-Algorithmic-Framework-for-F-Theory-Models-in-OSCAR)
    - [3.1 Weierstrass Models](#3.1:-Weierstrass-Models)
    - [3.2 The Refined Tate Fiber Type](#3.2:-The-Refined-Tate-Fiber-Type)
    - [3.3 Global Tate Models](#3.3:-Global-Tate-Models)
    - [3.4 Hypersurface Models](#3.4:-Hypersurface-Models)
    - [3.5 Resolution of Singularities --- Toric Blowups and Beyond](#3.5:-Resolution-of-Singularities-----Toric-Blowups-and-Beyond)
- [4. Literature Models](#4:-Literature-Models)
    - [4.1 Fundamentals of Literature Models](#4.1:-Fundamentals-of-Literature-Models)
    - [4.1 Toric Hypersurface Fibrations](#4.2:-Toric-Hypersurface-Fibrations)
- [5. Enumeration of G4 Fluxes](#5:-Enumeration-of-G4-Fluxes)
    - [5.1 Computation of G4-Fluxes](#5.1:-Computation-of-G4-Fluxes)
    - [5.2 G4-Fluxes in a Complex Model --- A Computational Stress Test](#5.2:-G4-Fluxes-in-a-Complex-Model-----A-Computational-Stress-Test)

### Warning on the code snippets in section 5.2

Those computations are memory-intensive and may take a significant amount of time to execute. To avoid unnecessary delays when running the entire notebook, we have commented out all code snippets in section 5.2. If you are interested in exploring these examples, uncomment the relevant lines.

In [1]:
using Oscar

In [2]:
using Test

## 3: Algorithmic Framework for F-Theory Models in OSCAR

### 3.1: Weierstrass Models

In [3]:
B2 = projective_space(NormalToricVariety, 2)

Normal toric variety

In [4]:
x1, x2, x3 = gens(cox_ring(B2))

3-element Vector{MPolyDecRingElem{QQFieldElem, QQMPolyRingElem}}:
 x1
 x2
 x3

In [5]:
weier_f = 1//48*(-(28*x1*x2^5 + 169*x3^6)^2 + 24*(2*x1^12 + 13*x1^2*x2^4*x3^6));
weier_g = 1//864*(216*x1^4*x2^8*x3^6 + (28*x1*x2^5 + 169*x3^6)^3 - 36*(28*x1*x2^5 + 169*x3^6)*(2*x1^12 + 13*x1^2*x2^4*x3^6));
w = weierstrass_model(B2, weier_f, weier_g; completeness_check = false)

Weierstrass model over a concrete base

In [6]:
explicit_model_sections(w)

Dict{String, MPolyDecRingElem{QQFieldElem, QQMPolyRingElem}} with 2 entries:
  "f" => x1^12 - 49//3*x1^2*x2^10 + 13//2*x1^2*x2^4*x3^6 - 1183//6*x1*x2^5*x3^6 - 28561//48*x3^12
  "g" => -7//3*x1^13*x2^5 - 169//12*x1^12*x3^6 + 1//4*x1^4*x2^8*x3^6 + 686//27*x1^3*x2^15 - 91//6*x1^3*x2^9*x3^6 + 8281//18*x1^2*x2^10*x3^6 - 2197//24*x1^2*x2^4*x3^12 + 199927//72*x1*x2^5*x3^12 + 4826809//864*x3^18

In [7]:
w_generic = weierstrass_model(B2; completeness_check = false)

Weierstrass model over a concrete base

In [8]:
explicit_model_sections(w_generic)

Dict{String, MPolyDecRingElem{QQFieldElem, QQMPolyRingElem}} with 2 entries:
  "f" => 6771*x1^12 - 8276*x1^11*x2 - 9725*x1^11*x3 - 1266*x1^10*x2^2 + 7173*x1^10*x2*x3 + 6269*x1^10*x3^2 - 6699*x1^9*x2^3 + 4998*x1^9*x2^2*x3 - 4438*x1^9*x2*x3^2 + 2337*x1^9*x3^3 - 2324*x1^8*x2^4 - 8871*x1^8*x2^3*x3 - 9620*x1^8*x2^2*…
  "g" => -2130*x1^18 + 4976*x1^17*x2 + 9222*x1^17*x3 - 7638*x1^16*x2^2 + 276*x1^16*x2*x3 + 3466*x1^16*x3^2 + 7461*x1^15*x2^3 - 4226*x1^15*x2^2*x3 + 8845*x1^15*x2*x3^2 + 778*x1^15*x3^3 + 6191*x1^14*x2^4 - 5246*x1^14*x2^3*x3 - 591*x1^14*…

### 3.2: The Refined Tate Fiber Type

In [9]:
singular_loci(w_generic)

1-element Vector{Tuple{MPolyIdeal{<:MPolyRingElem}, Tuple{Int64, Int64, Int64}, String}}:
 (Ideal with 1 generator, (0, 0, 1), "I_1")

In [10]:
singular_loci(w)

2-element Vector{Tuple{MPolyIdeal{<:MPolyRingElem}, Tuple{Int64, Int64, Int64}, String}}:
 (Ideal with 1 generator, (0, 0, 1), "I_1")
 (Ideal (x1), (0, 0, 5), "Non-split I_5")

### 3.3: Global Tate Models

In [11]:
a1 = 13 * x3^3;
a2 = 7 * x1 * x2^5;
a3 = x1^2 * x2^4 * x3^3;
a4 = x1^3 * (x2 + x3)^9;
a6 = zero(cox_ring(B2));
t = global_tate_model(B2, [a1, a2, a3, a4, a6])

Global Tate model over a concrete base

In [12]:
hypersurface_equation(t)

x1^3*x2^9*x*z^4 + 9*x1^3*x2^8*x3*x*z^4 + 36*x1^3*x2^7*x3^2*x*z^4 + 84*x1^3*x2^6*x3^3*x*z^4 + 126*x1^3*x2^5*x3^4*x*z^4 + 126*x1^3*x2^4*x3^5*x*z^4 + 84*x1^3*x2^3*x3^6*x*z^4 + 36*x1^3*x2^2*x3^7*x*z^4 + 9*x1^3*x2*x3^8*x*z^4 + x1^3*x3^9*x*z^4 - x1^2*x2^4*x3^3*y*z^3 + 7*x1*x2^5*x^2*z^2 - 13*x3^3*x*y*z + x^3 - y^2

In [13]:
w = weierstrass_model(t)

Weierstrass model over a concrete base

In [14]:
weierstrass_section_f(w)

x1^3*x2^9 + 9*x1^3*x2^8*x3 + 36*x1^3*x2^7*x3^2 + 84*x1^3*x2^6*x3^3 + 126*x1^3*x2^5*x3^4 + 126*x1^3*x2^4*x3^5 + 84*x1^3*x2^3*x3^6 + 36*x1^3*x2^2*x3^7 + 9*x1^3*x2*x3^8 + x1^3*x3^9 - 49//3*x1^2*x2^10 + 13//2*x1^2*x2^4*x3^6 - 1183//6*x1*x2^5*x3^6 - 28561//48*x3^12

In [15]:
weierstrass_section_g(w)

-7//3*x1^4*x2^14 - 21*x1^4*x2^13*x3 - 84*x1^4*x2^12*x3^2 - 196*x1^4*x2^11*x3^3 - 294*x1^4*x2^10*x3^4 - 294*x1^4*x2^9*x3^5 - 783//4*x1^4*x2^8*x3^6 - 84*x1^4*x2^7*x3^7 - 21*x1^4*x2^6*x3^8 - 7//3*x1^4*x2^5*x3^9 + 686//27*x1^3*x2^15 - 117//4*x1^3*x2^9*x3^6 - 507//4*x1^3*x2^8*x3^7 - 507*x1^3*x2^7*x3^8 - 1183*x1^3*x2^6*x3^9 - 3549//2*x1^3*x2^5*x3^10 - 3549//2*x1^3*x2^4*x3^11 - 1183*x1^3*x2^3*x3^12 - 507*x1^3*x2^2*x3^13 - 507//4*x1^3*x2*x3^14 - 169//12*x1^3*x3^15 + 8281//18*x1^2*x2^10*x3^6 - 2197//24*x1^2*x2^4*x3^12 + 199927//72*x1*x2^5*x3^12 + 4826809//864*x3^18

### 3.4: Hypersurface Models

In [16]:
fiber_ambient_space = weighted_projective_space(NormalToricVariety, [2,3,1])

Normal toric variety

In [17]:
set_coordinate_names(fiber_ambient_space, ["x" , "y" , "z" ])

In [18]:
D1 = 2 * anticanonical_divisor_class(B2)
D2 = 3 * anticanonical_divisor_class(B2)

Divisor class on a normal toric variety

In [19]:
amb_ring, (x1, x2, x3, x, y, z) = polynomial_ring(QQ, ["x1" , "x2" , "x3" , "x" , "y" , "z" ])

(Multivariate polynomial ring in 6 variables over QQ, QQMPolyRingElem[x1, x2, x3, x, y, z])

In [20]:
p = x^3 + 7*x1*x2^5*x^2*z^2 + x1^3*(x2 + x3)^9*x*z^4 - y^2 - 13*x3^3*x*y*z - x1^2*x2^4*x3^3*y*z^3

x1^3*x2^9*x*z^4 + 9*x1^3*x2^8*x3*x*z^4 + 36*x1^3*x2^7*x3^2*x*z^4 + 84*x1^3*x2^6*x3^3*x*z^4 + 126*x1^3*x2^5*x3^4*x*z^4 + 126*x1^3*x2^4*x3^5*x*z^4 + 84*x1^3*x2^3*x3^6*x*z^4 + 36*x1^3*x2^2*x3^7*x*z^4 + 9*x1^3*x2*x3^8*x*z^4 + x1^3*x3^9*x*z^4 - x1^2*x2^4*x3^3*y*z^3 + 7*x1*x2^5*x^2*z^2 - 13*x3^3*x*y*z + x^3 - y^2

In [21]:
h = hypersurface_model(B2, fiber_ambient_space, [D1, D2], p, completeness_check = false)

Hypersurface model over a concrete base

In [22]:
hypersurface_equation(h)

x1^3*x2^9*x*z^4 + 9*x1^3*x2^8*x3*x*z^4 + 36*x1^3*x2^7*x3^2*x*z^4 + 84*x1^3*x2^6*x3^3*x*z^4 + 126*x1^3*x2^5*x3^4*x*z^4 + 126*x1^3*x2^4*x3^5*x*z^4 + 84*x1^3*x2^3*x3^6*x*z^4 + 36*x1^3*x2^2*x3^7*x*z^4 + 9*x1^3*x2*x3^8*x*z^4 + x1^3*x3^9*x*z^4 - x1^2*x2^4*x3^3*y*z^3 + 7*x1*x2^5*x^2*z^2 - 13*x3^3*x*y*z + x^3 - y^2

In [23]:
tate_polynomial(t)

x1^3*x2^9*x*z^4 + 9*x1^3*x2^8*x3*x*z^4 + 36*x1^3*x2^7*x3^2*x*z^4 + 84*x1^3*x2^6*x3^3*x*z^4 + 126*x1^3*x2^5*x3^4*x*z^4 + 126*x1^3*x2^4*x3^5*x*z^4 + 84*x1^3*x2^3*x3^6*x*z^4 + 36*x1^3*x2^2*x3^7*x*z^4 + 9*x1^3*x2*x3^8*x*z^4 + x1^3*x3^9*x*z^4 - x1^2*x2^4*x3^3*y*z^3 + 7*x1*x2^5*x^2*z^2 - 13*x3^3*x*y*z + x^3 - y^2

In [24]:
cox_ring(ambient_space(t))

Multivariate polynomial ring in 6 variables over QQ graded by
  x1 -> [1 0]
  x2 -> [1 0]
  x3 -> [1 0]
  x -> [6 2]
  y -> [9 3]
  z -> [0 1]

In [25]:
cox_ring(ambient_space(h))

Multivariate polynomial ring in 6 variables over QQ graded by
  x1 -> [1 0]
  x2 -> [1 0]
  x3 -> [1 0]
  x -> [6 2]
  y -> [9 3]
  z -> [0 1]

### 3.5: Resolution of Singularities --- Toric Blowups and Beyond

In [26]:
B3 = projective_space(NormalToricVariety, 3)

Normal toric variety

In [27]:
cox_ring(B3)

Multivariate polynomial ring in 4 variables over QQ graded by
  x1 -> [1]
  x2 -> [1]
  x3 -> [1]
  x4 -> [1]

In [28]:
W = toric_line_bundle(torusinvariant_prime_divisors(B3)[1])

Toric line bundle on a normal toric variety

In [29]:
w = generic_section(W)

6149*x1 - 5412*x2 + 8001*x3 - 6399*x4

In [30]:
w = gens(cox_ring(B3))[1]

x1

In [31]:
Kbar = anticanonical_bundle(B3)

Toric line bundle on a normal toric variety

In [32]:
a10 = generic_section(Kbar);
a21 = generic_section(Kbar^2*W^(-1));
a32 = generic_section(Kbar^3*W^(-2));
a43 = generic_section(Kbar^4*W^(-3));
a6 = zero(cox_ring(B3));
t = global_tate_model(B3, [a10, a21 * w, a32 * w^2, a43 * w^3, a6])

Global Tate model over a concrete base

In [33]:
singular_loci(t)

2-element Vector{Tuple{MPolyIdeal{<:MPolyRingElem}, Tuple{Int64, Int64, Int64}, String}}:
 (Ideal with 1 generator, (0, 0, 1), "I_1")
 (Ideal (x1), (0, 0, 5), "Split I_5")

In [34]:
amb = ambient_space(t)

Normal toric variety

In [35]:
cox_ring(amb)

Multivariate polynomial ring in 7 variables over QQ graded by
  x1 -> [1 0]
  x2 -> [1 0]
  x3 -> [1 0]
  x4 -> [1 0]
  x -> [8 2]
  y -> [12 3]
  z -> [0 1]

In [36]:
t1 = blow_up(t, ["x" , "y" , string(w)]; coordinate_name = "e1" )

Partially resolved global Tate model over a concrete base

In [37]:
@test_throws ArgumentError resolutions(t)

Test Passed
      Thrown: ArgumentError

In [38]:
add_resolution!(t, [["x", "y", "w"], ["y", "e1"], ["x", "e4"], ["y", "e2"], ["x", "y"]], ["e1", "e4", "e2", "e3", "s"])

In [39]:
explicit_model_sections(t)["w" ] = w

x1

In [40]:
t_res = resolve(t, 1)

Partially resolved global Tate model over a concrete base

In [41]:
tate_polynomial(t_res)

257*x1^16*x*z^4*e1^15*e4^14*e2^14*e3^13 + 8141*x1^15*x2*x*z^4*e1^14*e4^13*e2^13*e3^12 - 5732*x1^15*x3*x*z^4*e1^14*e4^13*e2^13*e3^12 + 8418*x1^15*x4*x*z^4*e1^14*e4^13*e2^13*e3^12 - 3656*x1^14*x2^2*x*z^4*e1^13*e4^12*e2^12*e3^11 - 6680*x1^14*x2*x3*x*z^4*e1^13*e4^12*e2^12*e3^11 + 1485*x1^14*x2*x4*x*z^4*e1^13*e4^12*e2^12*e3^11 - 9892*x1^14*x3^2*x*z^4*e1^13*e4^12*e2^12*e3^11 - 3256*x1^14*x3*x4*x*z^4*e1^13*e4^12*e2^12*e3^11 + 5899*x1^14*x4^2*x*z^4*e1^13*e4^12*e2^12*e3^11 + 1431*x1^13*x2^3*x*z^4*e1^12*e4^11*e2^11*e3^10 - 4590*x1^13*x2^2*x3*x*z^4*e1^12*e4^11*e2^11*e3^10 - 1961*x1^13*x2^2*x4*x*z^4*e1^12*e4^11*e2^11*e3^10 - 6523*x1^13*x2*x3^2*x*z^4*e1^12*e4^11*e2^11*e3^10 + 831*x1^13*x2*x3*x4*x*z^4*e1^12*e4^11*e2^11*e3^10 - 4862*x1^13*x2*x4^2*x*z^4*e1^12*e4^11*e2^11*e3^10 + 7163*x1^13*x3^3*x*z^4*e1^12*e4^11*e2^11*e3^10 - 4137*x1^13*x3^2*x4*x*z^4*e1^12*e4^11*e2^11*e3^10 + 4140*x1^13*x3*x4^2*x*z^4*e1^12*e4^11*e2^11*e3^10 + 7236*x1^13*x4^3*x*z^4*e1^12*e4^11*e2^11*e3^10 + 6680*x1^12*x2^4*x*z^4*e1^11*

In [42]:
W = toric_line_bundle(2 * torusinvariant_prime_divisors(B3)[1])

Toric line bundle on a normal toric variety

In [43]:
w = generic_section(W)

6777*x1^2 + 1819*x1*x2 + 1037*x1*x3 - 7685*x1*x4 - 6425*x2^2 - 6666*x2*x3 - 7324*x2*x4 + 4609*x3^2 + 9846*x3*x4 - 3723*x4^2

In [44]:
a10 = generic_section(Kbar);
a21 = generic_section(Kbar^2*W^(-1));
a32 = generic_section(Kbar^3*W^(-2));
a43 = generic_section(Kbar^4*W^(-3));
a6 = zero(cox_ring(B3));
t2 = global_tate_model(B3, [a10, a21 * w, a32 * w^2, a43 * w^3, a6])

Global Tate model over a concrete base

In [45]:
singular_loci(t2)

2-element Vector{Tuple{MPolyIdeal{<:MPolyRingElem}, Tuple{Int64, Int64, Int64}, String}}:
 (Ideal with 1 generator, (0, 0, 1), "I_1")
 (Ideal with 1 generator, (0, 0, 5), "Split I_5")

In [46]:
singular_loci(t2)[2][1]

Ideal generated by
  6777*x1^2 + 1819*x1*x2 + 1037*x1*x3 - 7685*x1*x4 - 6425*x2^2 - 6666*x2*x3 - 7324*x2*x4 + 4609*x3^2 + 9846*x3*x4 - 3723*x4^2

In [47]:
t2_1 = blow_up(t, ["x" , "y" , string(w)]; coordinate_name = "e1" )

Partially resolved global Tate model over a concrete base

In [48]:
typeof(ambient_space(t2_1))

CoveredScheme{QQField}

In [49]:
ambient_space(t2_1)

Scheme
  over rational field
with default covering
  described by patches
     1: scheme(-(s1//s0)*x_5_1 + x_4_1, 6777*(s1//s0)*x_1_1^2 + 1819*(s1//s0)*x_1_1*x_2_1 + 1037*(s1//s0)*x_1_1*x_3_1 - 7685*(s1//s0)*x_1_1 - 6425*(s1//s0)*x_2_1^2 - 6666*(s1//s0)*x_2_1*x_3_1 - 7324*(s1//s0)*x_2_1 + 4609*(s1//s0)*x_3_1^2 + 9846*(s1//s0)*x_3_1 - 3723*(s1//s0) - (s2//s0)*x_4_1, -(s2//s0)*x_5_1 + 6777*x_1_1^2 + 1819*x_1_1*x_2_1 + 1037*x_1_1*x_3_1 - 7685*x_1_1 - 6425*x_2_1^2 - 6666*x_2_1*x_3_1 - 7324*x_2_1 + 4609*x_3_1^2 + 9846*x_3_1 - 3723)
     2: scheme((s0//s1)*x_4_1 - x_5_1, -(s2//s1)*x_4_1 + 6777*x_1_1^2 + 1819*x_1_1*x_2_1 + 1037*x_1_1*x_3_1 - 7685*x_1_1 - 6425*x_2_1^2 - 6666*x_2_1*x_3_1 - 7324*x_2_1 + 4609*x_3_1^2 + 9846*x_3_1 - 3723, 6777*(s0//s1)*x_1_1^2 + 1819*(s0//s1)*x_1_1*x_2_1 + 1037*(s0//s1)*x_1_1*x_3_1 - 7685*(s0//s1)*x_1_1 - 6425*(s0//s1)*x_2_1^2 - 6666*(s0//s1)*x_2_1*x_3_1 - 7324*(s0//s1)*x_2_1 + 4609*(s0//s1)*x_3_1^2 + 9846*(s0//s1)*x_3_1 - 3723*(s0//s1) - (s2//s1)*x_5_1)
     3: s

In [50]:
add_resolution!(t2, [["x", "y", "w"], ["y", "e1"], ["x", "e4"], ["y", "e2"], ["x", "y"]], ["e1", "e4", "e2", "e3", "s"])

In [51]:
explicit_model_sections(t2)["w" ] = w;

In [52]:
t2_res = resolve(t2, 1)

Partially resolved global Tate model over a concrete base

In [53]:
@test_throws ArgumentError tate_polynomial(t2_res)

Test Passed
      Thrown: ArgumentError

In [54]:
tate_ideal_sheaf(t2_res)

Sheaf of ideals
  on scheme over QQ covered with 48 patches
     1: [(s1//s0), (s1//s0), (s2//s0), x_1_1, x_2_1, x_3_1]   scheme(0, -(s1//s0)*(s1//s0)^2*(s2//s0) + 6777*(s1//s0)*x_1_1^2 + 1819*(s1//s0)*x_1_1*x_2_1 + 1037*(s1//s0)*x_1_1*x_3_1 - 7685*(s1//s0)*x_1_1 - 6425*(s1//s0)*x_2_1^2 - 6666*(s1//s0)*x_2_1*x_3_1 - 7324*(s1//s0)*x_2_1 + 4609*(s1//s0)*x_3_1^2 + 9846*(s1//s0)*x_3_1 - 3723*(s1//s0), -(s1//s0)*(s1//s0)*(s2//s0) + 6777*x_1_1^2 + 1819*x_1_1*x_2_1 + 1037*x_1_1*x_3_1 - 7685*x_1_1 - 6425*x_2_1^2 - 6666*x_2_1*x_3_1 - 7324*x_2_1 + 4609*x_3_1^2 + 9846*x_3_1 - 3723, 0, 0)
     2: [(s0//s1), (s2//s0), x_1_1, x_2_1, x_3_1, x_5_1]      scheme(0, -(s0//s1)*(s2//s0)*x_5_1^2 + 6777*(s0//s1)*x_1_1^2*x_5_1 + 1819*(s0//s1)*x_1_1*x_2_1*x_5_1 + 1037*(s0//s1)*x_1_1*x_3_1*x_5_1 - 7685*(s0//s1)*x_1_1*x_5_1 - 6425*(s0//s1)*x_2_1^2*x_5_1 - 6666*(s0//s1)*x_2_1*x_3_1*x_5_1 - 7324*(s0//s1)*x_2_1*x_5_1 + 4609*(s0//s1)*x_3_1^2*x_5_1 + 9846*(s0//s1)*x_3_1*x_5_1 - 3723*(s0//s1)*x_5_1, -(s2//s0)*x_5_1 + 

## 4: Literature Models

### 4.1: Fundamentals of Literature Models

In [55]:
B3 = projective_space(NormalToricVariety, 3)

Normal toric variety

In [56]:
W = torusinvariant_prime_divisors(B3)[1]

Torus-invariant, prime divisor on a normal toric variety

In [57]:
t = literature_model(arxiv_id = "1109.3454" , equation = "3.1" , base_space = B3, defining_classes = Dict("w" => W))

Global Tate model over a concrete base -- SU(5)xU(1) restricted Tate model based on arXiv paper 1109.3454 Eq. (3.1)

In [58]:
journal_name(t)

"Nucl. Phys. B"

In [59]:
paper_authors(t)

3-element Vector{String}:
 "Sven Krause"
 "Christoph Mayrhofer"
 "Timo Weigand"

In [60]:
arxiv_doi(t)

"10.48550/arXiv.1109.3454"

In [61]:
journal_model_equation_number(t)

"3.1"

In [62]:
journal_model_page(t)

"9"

In [63]:
defining_classes(t)

Dict{String, ToricDivisorClass} with 1 entry:
  "w" => Divisor class on a normal toric variety

In [64]:
model_sections(t)

9-element Vector{String}:
 "a21"
 "a6"
 "a3"
 "w"
 "a2"
 "a1"
 "a43"
 "a4"
 "a32"

In [65]:
model_section_parametrization(t)

Dict{String, MPolyRingElem} with 4 entries:
  "a6" => 0
  "a3" => w^2*a32
  "a2" => w*a21
  "a4" => w^3*a43

In [66]:
tunable_sections(t)

5-element Vector{String}:
 "a21"
 "w"
 "a1"
 "a43"
 "a32"

In [67]:
classes_of_model_sections(t)

Dict{String, ToricDivisorClass} with 9 entries:
  "a21" => Divisor class on a normal toric variety
  "a6"  => Divisor class on a normal toric variety
  "a3"  => Divisor class on a normal toric variety
  "w"   => Divisor class on a normal toric variety
  "a2"  => Divisor class on a normal toric variety
  "a1"  => Divisor class on a normal toric variety
  "a43" => Divisor class on a normal toric variety
  "a4"  => Divisor class on a normal toric variety
  "a32" => Divisor class on a normal toric variety

In [68]:
classes_of_tunable_sections_in_basis_of_Kbar_and_defining_classes(t)

Dict{String, Vector{Int64}} with 5 entries:
  "a21" => [2, -1]
  "w"   => [0, 1]
  "a1"  => [1, 0]
  "a43" => [4, -3]
  "a32" => [3, -2]

In [69]:
explicit_model_sections(t)

Dict{String, MPolyDecRingElem{QQFieldElem, QQMPolyRingElem}} with 9 entries:
  "a21" => 7600*x1^7 - 1002*x1^6*x2 - 1237*x1^6*x3 - 1993*x1^6*x4 - 9090*x1^5*x2^2 - 800*x1^5*x2*x3 + 6351*x1^5*x2*x4 + 8502*x1^5*x3^2 + 4283*x1^5*x3*x4 - 6289*x1^5*x4^2 + 2908*x1^4*x2^3 - 2037*x1^4*x2^2*x3 + 3241*x1^4*x2^2*x4 - 9589*x…
  "a6"  => 0
  "a3"  => 4712*x1^12 - 6036*x1^11*x2 + 9983*x1^11*x3 - 1496*x1^11*x4 - 5288*x1^10*x2^2 + 4780*x1^10*x2*x3 - 4602*x1^10*x2*x4 - 8403*x1^10*x3^2 - 4464*x1^10*x3*x4 - 9958*x1^10*x4^2 + 6585*x1^9*x2^3 + 3892*x1^9*x2^2*x3 + 9734*x1^9*x2^2*…
  "w"   => x1
  "a2"  => 7600*x1^8 - 1002*x1^7*x2 - 1237*x1^7*x3 - 1993*x1^7*x4 - 9090*x1^6*x2^2 - 800*x1^6*x2*x3 + 6351*x1^6*x2*x4 + 8502*x1^6*x3^2 + 4283*x1^6*x3*x4 - 6289*x1^6*x4^2 + 2908*x1^5*x2^3 - 2037*x1^5*x2^2*x3 + 3241*x1^5*x2^2*x4 - 9589*x…
  "a1"  => 7081*x1^4 - 3306*x1^3*x2 + 8830*x1^3*x3 - 3115*x1^3*x4 - 7343*x1^2*x2^2 - 5538*x1^2*x2*x3 + 2211*x1^2*x2*x4 + 4230*x1^2*x3^2 + 4419*x1^2*x3*x4 + 2909*x1^2*x4^2 - 6013*x1*x2^3

In [70]:
resolutions(t)

1-element Vector{Tuple{Vector{Vector{String}}, Vector{String}}}:
 ([["x", "y", "w"], ["y", "e1"], ["x", "e4"], ["y", "e2"], ["x", "y"]], ["e1", "e4", "e2", "e3", "s"])

In [71]:
t_res = resolve(t, 1)

Partially resolved global Tate model over a concrete base -- SU(5)xU(1) restricted Tate model based on arXiv paper 1109.3454 Eq. (3.1)

In [72]:
v = ambient_space(t_res);

In [73]:
cox_ring(v)

Multivariate polynomial ring in 12 variables over QQ graded by
  x1 -> [1 0 0 0 0 0 0]
  x2 -> [0 1 0 0 0 0 0]
  x3 -> [0 1 0 0 0 0 0]
  x4 -> [0 1 0 0 0 0 0]
  x -> [0 0 1 0 0 0 0]
  y -> [0 0 0 1 0 0 0]
  z -> [0 0 0 0 1 0 0]
  e1 -> [0 0 0 0 0 1 0]
  e4 -> [0 0 0 0 0 0 1]
  e2 -> [-1 -3 -1 1 -1 -1 0]
  e3 -> [0 4 1 -1 1 0 -1]
  s -> [2 6 -1 0 2 1 1]

In [74]:
M = matrix(map_from_torusinvariant_weil_divisor_group_to_class_group(v));
MT = transpose(M)

[1   0   0   0   0   0   0   0   0   -1    0    2]
[0   1   1   1   0   0   0   0   0   -3    4    6]
[0   0   0   0   1   0   0   0   0   -1    1   -1]
[0   0   0   0   0   1   0   0   0    1   -1    0]
[0   0   0   0   0   0   1   0   0   -1    1    2]
[0   0   0   0   0   0   0   1   0   -1    0    1]
[0   0   0   0   0   0   0   0   1    0   -1    1]

In [75]:
W = 2 * torusinvariant_prime_divisors(B3)[1]

Torus-invariant, non-prime divisor on a normal toric variety

In [76]:
t = literature_model(arxiv_id = "1109.3454" , equation = "3.1", base_space = B3, defining_classes = Dict("w" => W))

Global Tate model over a concrete base -- SU(5)xU(1) restricted Tate model based on arXiv paper 1109.3454 Eq. (3.1)

In [77]:
t_res = resolve(t, 1)

Partially resolved global Tate model over a concrete base -- SU(5)xU(1) restricted Tate model based on arXiv paper 1109.3454 Eq. (3.1)

### 4.2: Toric Hypersurface Fibrations

In [78]:
display_all_literature_models(Dict("gauge_algebra" => ["su(3)" , "su(2)", "u(1)"]))

Model 33:
("journal_section" => "3", "arxiv_page" => "67", "arxiv_id" => "1408.4808", "gauge_algebra" => Any["su(3)", "su(2)", "u(1)"], "arxiv_version" => "2", "journal_equation" => "3.141", "journal_page" => "67", "arxiv_equation" => "3.142", "journal_doi" => "10.1007/JHEP01(2015)142", "arxiv_section" => "3", "journal" => "JHEP", "file" => "model1408_4808-11-WSF.json", "arxiv_doi" => "10.48550/arXiv.1408.4808", "model_index" => "33", "type" => "weierstrass")

Model 34:
Dict{String, Any}("journal_section" => "3", "arxiv_page" => "67", "arxiv_id" => "1408.4808", "gauge_algebra" => Any["su(3)", "su(2)", "u(1)"], "arxiv_version" => "2", "journal_equation" => "3.141", "journal_page" => "67", "arxiv_equation" => "3.142", "journal_doi" => "10.1007/JHEP01(2015)142", "arxiv_section" => "3", "journal" => "JHEP", "file" => "model1408_4808-11.json", "arxiv_doi" => "10.48550/arXiv.1408.4808", "model_index" => "34", "type" => "hypersurface")

Model 39:
Dict{String, Any}("journal_section" => "3", "a

In [79]:
Kbar = anticanonical_divisor_class(B3)

Divisor class on a normal toric variety

In [80]:
foah11_B3 = literature_model(arxiv_id = "1408.4808" , equation = "3.142" , type = "hypersurface" , base_space = B3, 
        defining_classes = Dict("s7" => Kbar, "s9" => Kbar))

Hypersurface model over a concrete base

In [81]:
model_description(foah11_B3)

"F-theory hypersurface model with fiber ambient space F_11"

In [82]:
foah11_B3 = literature_model(34, base_space = B3, defining_classes =
Dict("s7" => Kbar, "s9" => Kbar))

Hypersurface model over a concrete base

In [83]:
zero_section(foah11_B3)

7-element Vector{MPolyDecRingElem{QQFieldElem, QQMPolyRingElem}}:
 1
 0
 -8871*x1^4 - 1622*x1^3*x2 - 1177*x1^3*x3 + 2224*x1^3*x4 - 9478*x1^2*x2^2 - 6306*x1^2*x2*x3 - 41*x1^2*x2*x4 - 6575*x1^2*x3^2 + 2042*x1^2*x3*x4 - 4121*x1^2*x4^2 - 9999*x1*x2^3 + 6004*x1*x2^2*x3 - 5656*x1*x2^2*x4 + 8072*x1*x2*x3^2 + 2437*x1*x2*x3*x4 - 7434*x1*x2*x4^2 + 8861*x1*x3^3 - 4912*x1*x3^2*x4 + 1182*x1*x3*x4^2 + 2141*x1*x4^3 + 8144*x2^4 + 3225*x2^3*x3 + 8733*x2^3*x4 + 8324*x2^2*x3^2 - 2866*x2^2*x3*x4 - 4648*x2^2*x4^2 - 4640*x2*x3^3 + 5627*x2*x3^2*x4 - 8233*x2*x3*x4^2 + 1922*x2*x4^3 - 4963*x3^4 + 8113*x3^3*x4 - 9517*x3^2*x4^2 + 9808*x3*x4^3 + 9031*x4^4
 1
 1
 6910*x1^4 - 844*x1^3*x2 + 6121*x1^3*x3 + 9429*x1^3*x4 + 3090*x1^2*x2^2 - 3556*x1^2*x2*x3 - 2393*x1^2*x2*x4 - 5655*x1^2*x3^2 + 442*x1^2*x3*x4 + 1662*x1^2*x4^2 - 1981*x1*x2^3 + 1230*x1*x2^2*x3 - 4545*x1*x2^2*x4 + 8980*x1*x2*x3^2 - 4717*x1*x2*x3*x4 - 9302*x1*x2*x4^2 + 4525*x1*x3^3 - 8259*x1*x3^2*x4 - 4130*x1*x3*x4^2 - 5583*x1*x4^3 + 8083*x2^4 - 2795*x2^3*x3 -

In [84]:
generating_sections(foah11_B3)

1-element Vector{Vector{MPolyDecRingElem{QQFieldElem, QQMPolyRingElem}}}:
 [-5004*x1^4 + 3507*x1^3*x2 - 3342*x1^3*x3 + 4149*x1^3*x4 + 8173*x1^2*x2^2 - 5178*x1^2*x2*x3 - 520*x1^2*x2*x4 + 2799*x1^2*x3^2 - 5609*x1^2*x3*x4 - 6650*x1^2*x4^2 - 3792*x1*x2^3 - 5505*x1*x2^2*x3 + 6048*x1*x2^2*x4 + 407*x1*x2*x3^2 + 6100*x1*x2*x3*x4 - 3674*x1*x2*x4^2 + 5925*x1*x3^3 + 2042*x1*x3^2*x4 + 7579*x1*x3*x4^2 - 6760*x1*x4^3 + 5662*x2^4 + 5065*x2^3*x3 + 4026*x2^3*x4 - 6206*x2^2*x3^2 + 5145*x2^2*x3*x4 - 8881*x2^2*x4^2 + 6994*x2*x3^3 - 8551*x2*x3^2*x4 + 5209*x2*x3*x4^2 + 7737*x2*x4^3 - 6804*x3^4 - 3246*x3^3*x4 + 5424*x3^2*x4^2 - 7268*x3*x4^3 + 8866*x4^4, 1, 1, -5986*x1^4 + 5773*x1^3*x2 - 1823*x1^3*x3 + 1577*x1^3*x4 - 2852*x1^2*x2^2 + 3629*x1^2*x2*x3 - 1113*x1^2*x2*x4 + 1937*x1^2*x3^2 - 3460*x1^2*x3*x4 - 3143*x1^2*x4^2 - 9338*x1*x2^3 - 2341*x1*x2^2*x3 + 7700*x1*x2^2*x4 + 4172*x1*x2*x3^2 + 5640*x1*x2*x3*x4 + 1566*x1*x2*x4^2 + 2874*x1*x3^3 + 2517*x1*x3^2*x4 - 2273*x1*x3*x4^2 + 9458*x1*x4^3 - 4600*x2^4 + 2134*x2^

In [85]:
g = gauge_algebra(foah11_B3)

Direct sum Lie algebra
  of dimension 12
with summands
  sl_3
  sl_2
  linear Lie algebra
over algebraic closure of rational field

In [86]:
global_gauge_group_quotient(foah11_B3)

3-element Vector{Vector{String}}:
 ["diagonal_matrix(root_of_unity(C,3),3)"]
 ["-identity_matrix(C,2)"]
 ["diagonal_matrix(root_of_unity(C,6,-1),1)"]

In [87]:
C = algebraic_closure(QQ)

Algebraic closure of rational field

In [88]:
diagonal_matrix(root_of_unity(C,3),3)

[{a2: -0.500 + 0.866*im}                   {a1: 0}                   {a1: 0}]
[                {a1: 0}   {a2: -0.500 + 0.866*im}                   {a1: 0}]
[                {a1: 0}                   {a1: 0}   {a2: -0.500 + 0.866*im}]

In [89]:
foah11_B3_weier = weierstrass_model(foah11_B3)

Weierstrass model over a concrete base -- F-theory weierstrass model dual to hypersurface model with fiber ambient space F_11 based on arXiv paper 1408.4808 Eq. (3.142)

In [90]:
birational_literature_models(foah11_B3)

1-element Vector{String}:
 "1408_4808-11-WSF"

In [91]:
singular_loci(foah11_B3_weier)

3-element Vector{Tuple{MPolyIdeal{<:MPolyRingElem}, Tuple{Int64, Int64, Int64}, String}}:
 (Ideal with 1 generator, (0, 0, 1), "I_1")
 (Ideal with 1 generator, (0, 0, 2), "Non-split I_2")
 (Ideal with 1 generator, (0, 0, 3), "Split I_3")

In [92]:
tunable_sections(foah11_B3_weier)

6-element Vector{String}:
 "s1"
 "s5"
 "s6"
 "s2"
 "s9"
 "s3"

In [93]:
R = cox_ring(base_space(foah11_B3_weier));
tuned_model = tune(foah11_B3_weier, Dict("s5" =>zero(R)))

Weierstrass model over a concrete base

In [94]:
singular_loci(tuned_model)

4-element Vector{Tuple{MPolyIdeal{<:MPolyRingElem}, Tuple{Int64, Int64, Int64}, String}}:
 (Ideal with 1 generator, (0, 0, 1), "I_1")
 (Ideal with 1 generator, (0, 0, 2), "Non-split I_2")
 (Ideal with 1 generator, (0, 0, 2), "Non-split I_2")
 (Ideal with 1 generator, (0, 0, 4), "Split I_4")

The following line triggers Julia's garbage collector to free memory from unused objects. This is mainly relevant for the OSCAR test suite, which verifies that tutorials run correctly with the latest OSCAR release. End users can safely ignore this line.

## 5: Enumeration of G4-Fluxes

### 5.1: Computation of G4-Fluxes

In [95]:
qsm_model = literature_model(arxiv_id = "1903.00009" , model_parameters = Dict("k" => 283))

Hypersurface model over a concrete base

In [96]:
cox_ring(ambient_space(qsm_model))

Multivariate polynomial ring in 14 variables over QQ graded by
  x1 -> [1 0 0 0 0 0 0 0 0]
  x2 -> [0 1 0 0 0 0 0 0 0]
  x3 -> [0 0 1 0 0 0 0 0 0]
  x4 -> [0 0 0 1 0 0 0 0 0]
  x5 -> [0 1 0 -1 0 0 0 0 0]
  x6 -> [1 0 0 0 0 0 0 0 0]
  x7 -> [-2 1 1 1 0 0 0 0 0]
  v -> [0 0 0 0 1 0 0 0 0]
  e3 -> [0 0 0 0 0 1 0 0 0]
  e2 -> [0 0 0 0 0 0 1 0 0]
  u -> [0 0 0 0 0 0 0 1 0]
  e4 -> [0 0 0 0 0 0 0 0 1]
  e1 -> [0 0 0 0 1 1 0 -1 -2]
  w -> [0 0 0 0 0 1 1 1 1]

In [97]:
stanley_reisner_ideal(ambient_space(qsm_model))

Ideal generated by
  x1*x6
  x2*x4
  x2*x5
  x3*x5
  x3*x7
  v*e2
  v*u
  v*e4
  v*e1
  e3*w
  e3*u
  e3*e4
  e3*e1
  x4*x7
  e2*w
  e2*e4
  e2*e1
  u*w
  u*e1
  e4*w

In [98]:
cohomology_ring(ambient_space(qsm_model))

Quotient
  of multivariate polynomial ring in 14 variables over QQ graded by
    x1 -> [1]
    x2 -> [1]
    x3 -> [1]
    x4 -> [1]
    x5 -> [1]
    x6 -> [1]
    x7 -> [1]
    v -> [1]
    e3 -> [1]
    e2 -> [1]
    u -> [1]
    e4 -> [1]
    e1 -> [1]
    w -> [1]
  by ideal with 25 generators

In [99]:
binomial(14,2)

91

In [100]:
betti_number(ambient_space(qsm_model), 4)

25

In [101]:
basis_of_h22_ambient(qsm_model, completeness_check = false)

25-element Vector{CohomologyClass}:
 Cohomology class on a normal toric variety given by x6*e4
 Cohomology class on a normal toric variety given by x4*x6
 Cohomology class on a normal toric variety given by x7^2
 Cohomology class on a normal toric variety given by x5*x6
 Cohomology class on a normal toric variety given by x6*x7
 Cohomology class on a normal toric variety given by x7*e2
 Cohomology class on a normal toric variety given by x7*u
 Cohomology class on a normal toric variety given by x7*e4
 Cohomology class on a normal toric variety given by x7*e1
 Cohomology class on a normal toric variety given by x7*w
 Cohomology class on a normal toric variety given by x5*u
 Cohomology class on a normal toric variety given by x4*e4
 Cohomology class on a normal toric variety given by x5*e4
 Cohomology class on a normal toric variety given by e1*w
 Cohomology class on a normal toric variety given by x6*e2
 Cohomology class on a normal toric variety given by x6*u
 Cohomology class on a nor

In [102]:
g4_gens = chosen_g4_flux_gens(qsm_model);
length(g4_gens)

25

In [103]:
g4_gens[1]

G4-flux candidate
  - Elementary quantization checks: not executed
  - Transversality checks: not executed
  - Non-abelian gauge group: breaking pattern not analyzed
  - Tadpole cancellation check: not computed

In [104]:
cohomology_class(g4_gens[1])

Cohomology class on a normal toric variety given by x6*e4

In [105]:
#Oscar._ambient_space_divisor_pairs_to_be_considered(qsm_model)

In [106]:
fg = special_flux_family(qsm_model, completeness_check = false)

Family of G4 fluxes:
  - Elementary quantization checks: satisfied
  - Transversality checks: satisfied
  - Non-abelian gauge group: breaking pattern not analyzed

In [107]:
mat_int = matrix_integral(fg);
size(mat_int)

(25, 17)

In [108]:
mat_rat = matrix_rational(fg);
size(mat_rat)

(25, 0)

In [109]:
g4 = random_flux_instance(fg)

G4-flux candidate
  - Elementary quantization checks: satisfied
  - Transversality checks: satisfied
  - Non-abelian gauge group: breaking pattern not analyzed
  - Tadpole cancellation check: not computed

In [110]:
g4_2 = random_flux(qsm_model, completeness_check = false)

G4-flux candidate
  - Elementary quantization checks: satisfied
  - Transversality checks: satisfied
  - Non-abelian gauge group: breaking pattern not analyzed
  - Tadpole cancellation check: not computed

In [111]:
fg_not_breaking = special_flux_family(qsm_model, not_breaking = true, completeness_check = false)

Family of G4 fluxes:
  - Elementary quantization checks: satisfied
  - Transversality checks: satisfied
  - Non-abelian gauge group: unbroken

In [112]:
size(matrix_integral(fg_not_breaking))

(25, 1)

In [113]:
b22 = chosen_g4_flux_gens(qsm_model);
mat_int = matrix_integral(fg_not_breaking);
g4_sample = sum([mat_int[k] * b22[k] for k in 1:length(b22)])

G4-flux candidate
  - Elementary quantization checks: not executed
  - Transversality checks: not executed
  - Non-abelian gauge group: breaking pattern not analyzed
  - Tadpole cancellation check: not computed

In [114]:
cohomology_class(g4_sample)

Cohomology class on a normal toric variety given by -1//5*x5*x6 + 1//30*x5*e2 + 1//15*x5*u + 1//10*x5*e4 + 1//15*x5*e1 + 1//30*x5*w - 11//15*x6*x7 + 2//15*x6*e2 + 4//15*x6*u + 2//5*x6*e4 + 4//15*x6*e1 + 2//15*x6*w - 7//30*x7^2 + 1//15*x7*e2 + 2//15*x7*u + 1//5*x7*e4 + 2//15*x7*e1 + 1//15*x7*w - 1//6*e1*w

In [115]:
kbar3(qsm_model)

30

In [116]:
divs = torusinvariant_prime_divisors(ambient_space(qsm_model));
e1 = cohomology_class(divs[13]);
e2 = cohomology_class(divs[10]);
e4 = cohomology_class(divs[12]);
u = cohomology_class(divs[11]);
v = cohomology_class(divs[8]);
pb_Kbar = cohomology_class(sum([divs[k] for k in 1:7]));

In [117]:
g4_class = (-3) // kbar3(qsm_model) * (5 * e1 * e4 + pb_Kbar * (-3 * e1 - 2 * e2 - 6 * e4 + pb_Kbar - 4 * u + v));

In [118]:
g4_sample2 = g4_flux(qsm_model, g4_class, completeness_check = false)

G4-flux candidate
  - Elementary quantization checks: satisfied
  - Transversality checks: satisfied
  - Non-abelian gauge group: breaking pattern not analyzed
  - Tadpole cancellation check: not computed

In [119]:
qsm_flux(qsm_model)

G4-flux candidate
  - Elementary quantization checks: satisfied
  - Transversality checks: satisfied
  - Non-abelian gauge group: unbroken
  - Tadpole cancellation check: not computed

In [120]:
 qsm_flux(qsm_model) == g4_sample2

true

In [121]:
cohomology_class(g4_sample2)

Cohomology class on a normal toric variety given by -3//5*x5*x6 + 1//10*x5*e2 + 1//5*x5*u + 3//10*x5*e4 + 1//5*x5*e1 + 1//10*x5*w - 11//5*x6*x7 + 2//5*x6*e2 + 4//5*x6*u + 6//5*x6*e4 + 4//5*x6*e1 + 2//5*x6*w - 7//10*x7^2 + 1//5*x7*e2 + 2//5*x7*u + 3//5*x7*e4 + 2//5*x7*e1 + 1//5*x7*w - 1//2*e1*w

In [122]:
3 * cohomology_class(g4_sample) - cohomology_class(g4_sample2)

Cohomology class on a normal toric variety given by 0

In [123]:
d3_tadpole_constraint(fg_not_breaking)

-1//12*a1^2 + 123//4

In [124]:
g4_exp = flux_instance(fg_not_breaking, [3], [])

G4-flux candidate
  - Elementary quantization checks: satisfied
  - Transversality checks: satisfied
  - Non-abelian gauge group: unbroken
  - Tadpole cancellation check: not computed

In [125]:
d3_tadpole_constraint(g4_exp)

30

In [126]:
d3_tadpole_constraint(g4_exp) == -1//12*3^2 + 123//4

true

### 5.2: G4-Fluxes in a Complex Model --- A Computational Stress Test

⚠️ Warning: The following computations are memory-intensive and may take a significant amount of time to execute.

To avoid unnecessary delays when running the entire notebook, we have commented them out by default.

If you are interested in exploring these examples, feel free to uncomment the lines you are curious about.

In [127]:
# t = literature_model(arxiv_id = "1511.03209")

In [128]:
# t_res = resolve(t, 1)

In [129]:
# amb = ambient_space(t_res);
# cohomology_ring(amb, check = false);

In [130]:
# betti_number(amb, 4)

In [131]:
# g4_amb_candidates = chosen_g4_flux_gens(t_res, completeness_check = false);

In [132]:
# length(g4_amb_candidates)

In [133]:
# fg_quant = special_flux_family(t_res, completeness_check = false, algorithm = "special")

In [134]:
# size(matrix_integral(fg_quant))

In [135]:
# size(matrix_rational(fg_quant))

In [136]:
# fg_quant_no_break = special_flux_family(t_res, not_breaking = true, completeness_check = false, algorithm = "special")

In [137]:
# size(matrix_integral(fg_quant_no_break))

In [138]:
# size(matrix_rational(fg_quant_no_break))